In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import plotly.express as px

In [2]:
results = pd.read_csv("data/processed/world_cup/all_editions/results.csv")
history = pd.read_csv("INT-World Cup/world_cup/fifa_world_cup_history.csv")
placement = pd.read_csv("INT-World Cup/world_cup/2002/placement.csv")
all_editions = pd.read_csv("data/processed/world_cup/all_editions/placement.csv")
pd.set_option("display.max_columns", None)

#display(results.info)
#display(placement.columns)
#display(placement["placement"].unique())

In [3]:
scope_map = {
    "UEFA": "europe",
    "CAF": "africa",
    "AFC": "asia",
    "CONMEBOL": "south america",
    "CONCACAF": "north america",
    "OFC": "world",  # Plotly has no clean Oceania scope
}

plot_df = maps_df[maps_df["confederation"].eq(selected_confederation)].copy()

fig3 = px.choropleth(
    plot_df,
    locations="fifa_code",      # or your ISO-3 column
    locationmode="ISO-3",
    color="country",
    hover_name="country",
    scope=scope_map[selected_confederation],
    title=f"{selected_confederation} World Cup Teams"
)

fig3.show()

NameError: name 'maps_df' is not defined

In [ ]:
display(results.columns)

Index(['edition', 'match_number', 'date', 'stage', 'home_team', 'away_team',
       'home_score', 'away_score', 'city', 'country', 'neutral',
       'decided_by_shootout', 'shootout_winner'],
      dtype='object')

In [ ]:
#Add next edition placement for all countries
print(history.columns)
print(all_editions.columns)

history = history.rename(columns={"Year": "edition", "Teams": "teams"})
all_editions = all_editions.merge(
   history[["edition", "teams"]],
    on="edition",
    how="left",
    validate="many_to_one"
    )


display(history)

Index(['Year', 'Host_Country', 'Winner', 'Runner_Up', 'Third_Place',
       'Fourth_Place', 'Total_Goals', 'Matches_Played', 'Teams',
       'Goals_Per_Match'],
      dtype='object')
Index(['edition', 'country', 'placement', 'position', 'matches_played', 'gs',
       'ga', 'next_edition', 'next_placement', 'next_position'],
      dtype='object')


,edition,Host_Country,Winner,Runner_Up,Third_Place,Fourth_Place,Total_Goals,Matches_Played,teams,Goals_Per_Match
0,1930,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,18,13,3.89
1,1934,Italy,Italy,Czechoslovakia,Germany,Austria,70,17,16,4.12
2,1938,France,Italy,Hungary,Brazil,Sweden,84,18,15,4.67
3,1950,Brazil,Uruguay,Brazil,Sweden,Spain,88,22,13,4.00
4,1954,Switzerland,West Germany,Hungary,Austria,Uruguay,140,26,16,5.38
5,1958,Sweden,Brazil,Sweden,France,West Germany,126,35,16,3.60
6,1962,Chile,Brazil,Czechoslovakia,Chile,Yugoslavia,89,32,16,2.78
7,1966,England,England,West Germany,Portugal,Soviet Union,89,32,16,2.78
8,1970,Mexico,Brazil,Italy,West Germany,Uruguay,95,32,16,2.97
9,1974,West Germany,West Germany,Netherlands,Poland,Brazil,97,38,16,2.55


In [ ]:
#Every countries Placement over the years

#Every countries Placement over the years for all World cup winners
df = all_editions.query("country == 'Germany'").sort_values("edition")
positions = ["position", "next_position"]
fig = px.bar(df, x="edition", y=positions, barmode='group')
fig.show()

In [ ]:
#Every countries Placement over the years for all World cup winners
L10 = all_editions.query("placement == 'Winner' and edition >= 1930").sort_values("edition")
positions = ["position", "next_position"]
fig = px.bar(df, x="edition", y=positions, barmode='group', title="Every World Cup Winners Placement the following Edition")
fig.update_yaxes(autorange="reversed")

fig.show()

In [ ]:
df = all_editions.query("placement == 'Winner'" ).copy()
df["edition"] = pd.to_numeric(df["edition"])
df["next_position"] = pd.to_numeric(df["next_position"], errors="coerce")
tick_vals = sorted(df["edition"].unique())

placement_map = {
    "Winner": "W",
    "Runner-up": "F",
    "Third Place": "3P",
    "Fourth Place": "4P",
    "Semi-final": "SF",
    "Quarter-final": "QF",
    "Round of 16": "R16",
    "Group Stage": "GS",
}

df["placement_short"] = df["placement"].map(placement_map)
df["next_placement_short"] = df["next_placement"].map(placement_map)

#min year
min_ed = df["edition"].min()

fig = px.bar(
    df,
    x="edition",
    y="next_position",
    hover_data=["country", "next_edition", "next_placement"],
    text="next_placement_short",
    title="Every World Cup Winners Placement the following Edition",
    color="country",

)
fig.update_yaxes(
    autorange="reversed",
    title= "Finish"
    )
fig.update_xaxes(
    tickmode="array",
    tickvals=tick_vals,
    ticktext=[str(x) for x in tick_vals],
    tickangle=30,
    title= "Tournament Edition"
)

fig.update_layout(
    height=600,
    width=1000,
     title=dict(
            font=dict(size=20, color="#3A2A1A"),
            text= f"World Cup Winners Placement the following Edition since {min_ed}",
            x=0.5
        ),
    legend = dict(
        title="Team"
        ),
    font=dict(
        family="Gill Sans, Arial, sans-serif",
        color="#3A2A1A",
        size=10,
    ),
    margin=dict(l=40, r=40, t=60, b=40),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='#5A4632',
        tickfont=dict(size=13, color="#5A4632"),
    ),

    yaxis=dict(
        title="Final Placement",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='#5A4632',
        tickfont=dict(size=13, color="#5A4632"),
    )
        
)

#add caption at the bottom of the chart
fig.add_annotation(
        text="Data Source: Kaggle | @cartierkut1",
        xref="paper", yref="paper",
        x=0, y=-0.1,
        showarrow=False,
        font=dict(size=10, color="#5A4632"),
)


fig.show(scale=3)
fig.write_image("assets/visualizations/b2b.png", scale=2)




In [ ]:
base = placement.copy()
base["edition"] = pd.to_numeric(base["edition"])
base["position"] = pd.to_numeric(base["position"], errors="coerce")

editions = sorted(base["edition"].unique())
next_edition_map = dict(zip(editions[:-1], editions[1:]))

# Winners only, excluding the 1930 13-team tournament for a fairer modern-format comparison.
df = base.loc[base["placement"].eq("Winner") & (base["teams"] > 15), ["edition", "country"]].copy()

# Minimum winner year in this filtered view.
min_ed = df["edition"].min()

df["next_edition"] = df["edition"].map(next_edition_map)

# Check the same team in the immediate next World Cup.
lookup = (
    base[["country", "edition", "placement", "position"]]
    .rename(
        columns={
            "edition": "next_edition",
            "placement": "next_placement",
            "position": "next_position",
        }
    )
)

df = df.merge(lookup, on=["country", "next_edition"], how="left")

# missing in the immediate next edition = did not qualify
df.loc[df["next_edition"].notna() & df["next_position"].isna(), "next_placement"] = "DNQ"

# no next edition available in the dataset yet
df.loc[df["next_edition"].isna(), "next_placement"] = "TBD"

dnq_rank = base["position"].max() + 1
tbd_rank = dnq_rank + 1
df["plot_position"] = df["next_position"]
df.loc[df["next_placement"].eq("DNQ"), "plot_position"] = dnq_rank
df.loc[df["next_placement"].eq("TBD"), "plot_position"] = tbd_rank

placement_map = {
    "Winner": "W",
    "Runner-up": "F",
    "Third Place": "3P",
    "Fourth Place": "4P",
    "Semi-final": "SF",
    "Quarter-final": "QF",
    "Round of 16": "R16",
    "Group Stage": "GS",
    "DNQ": "DNQ",
    "TBD": "TBD",
}

value_map = {
    "W": 1,
    "F": 2,
    "3P": 3,
    "4P": 4,
    "SF": 4,
    "QF": 8,
    "R16": 16,
    "GS": 32,
    "DNQ": dnq_rank,
    "TBD": pd.NA,
}

df["next_placement_short"] = df["next_placement"].map(placement_map)
df["next_placement_value"] = df["next_placement_short"].map(value_map)

outcome_colors = {
    "W": "#2F6F73",
    "F": "#C99700",
    "3P": "#7A4E2D",
    "4P": "#A35D2B",
    "SF": "#8A3FFC",
    "QF": "#D1495B",
    "R16": "#6F6A5F",
    "GS": "#9A8C7A",
    "DNQ": "#3A2A1A",
    "TBD": "#B8AA96",
}

outcome_order = ["W", "F", "3P", "4P", "SF", "QF", "R16", "GS", "DNQ", "TBD"]
outcome_labels = {
    "W": "Defended title",
    "F": "Runner-up",
    "3P": "Third place",
    "4P": "Fourth place",
    "SF": "Semi-final",
    "QF": "Quarter-final",
    "R16": "Round of 16",
    "GS": "Group stage",
    "DNQ": "Did not qualify",
    "TBD": "Next edition TBD",
}

fig = go.Figure()
for outcome in outcome_order:
    df_outcome = df[df["next_placement_short"].eq(outcome)]
    if df_outcome.empty:
        continue

    fig.add_trace(go.Scatter(
        x=df_outcome["edition"],
        y=df_outcome["plot_position"],
        mode="markers+text",
        name=outcome_labels[outcome],
        text=df_outcome["next_placement_short"],
        textposition="top center",
        marker=dict(
            color=outcome_colors[outcome],
            size=18,
            line=dict(color="#EFE3CF", width=1.5),
        ),
        customdata=df_outcome[["country", "next_edition", "next_placement"]],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Won: %{x}<br>"
            "Next World Cup: %{customdata[1]}<br>"
            "Next finish: %{customdata[2]}"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f"World Cup Winners Placement the following Edition since {min_ed}",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    font=dict(
        family="Inter, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=40, r=40, t=70, b=60),
    showlegend=True,
    legend=dict(
        title="Next World Cup Result",
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.08,
        yanchor="bottom",
        font=dict(size=10, color="#3A2A1A"),
    ),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Edition won by defending champion",
        tickmode="array",
        tickvals=df["edition"],
        ticktext=[str(x) for x in df["edition"]],
        showgrid=False,
        zeroline=False,
        showline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
        tickangle=30,
    ),
    yaxis=dict(
        title="Finish at next World Cup",
        autorange="reversed",
        tickmode="array",
        tickvals=[1, 2, 3, 4, 8, 16, 32, dnq_rank, tbd_rank],
        ticktext=["Winner", "Runner-up", "3rd", "4th", "QF", "R16", "Group", "DNQ", "TBD"],
        showgrid=True,
        gridcolor="#D8C8AF",
        zeroline=False,        
        showline=False,
        showticklabels=True,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
    annotations=[
        dict(
            text="Data Source: Kaggle | @cartierkut1",
            x=0, y=-0.1,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10, color="#5A4632"),
        ),
    ],
)


fig.show(scale=3)
fig.write_image("assets/visualizations/b2b3.png", scale=2)

print(df["next_placement_short"].value_counts())
print(df["edition"].count())
print(df["next_placement_value"].mean())


In [ ]:
fig.update_traces(
    textfont=dict(
        size=11,
        family="Gill Sans, Arial, sans-serif",
        color="#3A2A1A",
    ),
    hovertemplate="<b>%{y}</b><br>%{x}: %{z} matches<extra></extra>",
)

fig.update_layout(
    width=600,
    height=1000,
    autosize=False,

    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",

    title=dict(
        text=f"World Cup Winners Placement the following Edition since {min_ed}",
        x=0.5,
        y=0.96,
        font=dict(
            family="Gill Sans, Arial, sans-serif",
            size=24,
            color="#2F2418",
        ),
    ),

    font=dict(
        family="Gill Sans, Arial, sans-serif",
        size=12,
        color="#4A3A29",
    ),

    margin=dict(l=130, r=70, t=90, b=80),

    xaxis=dict(
        title="",
        side="top",
        showgrid=False,
        zeroline=False,
        showline=False,
        tickfont=dict(size=13, color="#5A4632"),
    ),

    yaxis=dict(
        title="",
        showgrid=False,
        zeroline=False,
        showline=False,
        tickfont=dict(size=12, color="#5A4632"),
    ),

    coloraxis_colorbar=dict(
    title=dict(
        text="Matches",
        font=dict(color="#5A4632", size=12),
    ),
    thickness=12,
    len=0.55,
    outlinewidth=0,
    tickfont=dict(color="#5A4632"),
),
)

In [ ]:
base = all_editions.copy()
base["edition"] = pd.to_numeric(base["edition"])
base["position"] = pd.to_numeric(base["position"], errors="coerce")

editions = sorted(base["edition"].unique())
next_edition_map = dict(zip(editions[:-1], editions[1:]))

# winners only
df = base.loc[base["placement"].eq("Winner") & (base["edition"] >= 1950), ["edition", "country"]].copy()
df["next_edition"] = df["edition"].map(next_edition_map)

#min year
min_ed = df["edition"].min()

# lookup that checks the same team in the immediate next World Cup
lookup = (
    base[["country", "edition", "placement", "position"]]
    .rename(
        columns={
            "edition": "next_edition",
            "placement": "next_placement",
            "position": "next_position",
        }
    )
)

df = df.merge(lookup, on=["country", "next_edition"], how="left")

# missing in the immediate next edition = did not qualify
df.loc[df["next_edition"].notna() & df["next_position"].isna(), "next_placement"] = "DNQ"

# no next edition available in the dataset yet
df.loc[df["next_edition"].isna(), "next_placement"] = "TBD"

dnq_rank = base["position"].max() + 1
df["plot_position"] = df["next_position"]
df.loc[df["next_placement"].eq("DNQ"), "plot_position"] = dnq_rank

placement_map = {
    "Winner": "W",
    "Runner-up": "F",
    "Third Place": "3P",
    "Fourth Place": "4P",
    "Semi-final": "SF",
    "Quarter-final": "QF",
    "Round of 16": "R16",
    "Group Stage": "GS",
    "DNQ": "DNQ",
    "TBD": "TBD",
}

value_map = {
    "W": 1,
    "F": 2,
    "3P": 3,
    "4P": 4,
    "QF": 8,
    "R16": 16,
    "GS": 32,
    }

df["next_placement_short"] = df["next_placement"].map(placement_map)
df["next_placement_value"] = df["next_placement_short"].map(value_map)
fig = px.bar(
    df,
    x="edition",
    y="plot_position",
    hover_data=["country", "next_edition", "next_placement"],
    text="next_placement_short",
    title="Every World Cup Winners Placement the following Edition",
    color="country",

)
fig.update_yaxes(
    autorange="reversed",
    title= "Finish",
    tickvals=[1, 2, 3, 4, 8, 16, 32, dnq_rank],
    ticktext=["W", "F", "3P", "4P", "QF", "R16", "GS", "DNQ"],
    )
fig.update_xaxes(
    tickmode="array",
    tickvals=tick_vals,
    ticktext=[str(x) for x in tick_vals],
    tickangle=45,
    title= "Tournament Edition"
)

fig.update_layout(
    height=600,
    width=1000,
     title=dict(
            font=dict(size=20, color="Beige"),
            text= f"World Cup Winners Placement the following Edition since {min_ed}",
            x=0.5
        ),
    legend = dict(
        title="Team"
        ),
    font=dict(
        family="Courier New, monospace",
        size=10,
        color="Beige"
    ),
    margin=dict(l=40, r=40, t=60, b=40),
    paper_bgcolor="#0B0F0C", autosize=False, plot_bgcolor = '#0B0F0C',
    xaxis=dict(
        title="Tournament Edition",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='Beige',
        tickcolor= 'Beige',
    ),

    yaxis=dict(
        title="Final Placement",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='Beige',
        tickcolor= 'Beige',
    )
        
)

#add caption at the bottom of the chart
fig.add_annotation(
        text="Data Source: Kaggle | @cartierkut1",
        xref="paper", yref="paper",
        x=0, y=-0.1,
        showarrow=False,
        font=dict(size=10, color="Beige"),
)


fig.show(scale=3)
fig.write_image("assets/visualizations/b2b2.png", scale=2)

print(df["next_placement_short"].value_counts())
print(df["next_placement_value"].mean())


next_placement_short
GS     5
QF     5
4P     3
F      3
W      1
R16    1
TBD    1
Name: count, dtype: int64
13.055555555555555


In [ ]:
base = all_editions.copy()
base["edition"] = pd.to_numeric(base["edition"])
base["position"] = pd.to_numeric(base["position"], errors="coerce")

editions = sorted(base["edition"].unique())
next_edition_map = dict(zip(editions[:-1], editions[1:]))

# winners only
df = base.loc[base["placement"].eq("Winner") & (base["teams"] > 15), ["edition", "country"]].copy()

#min year
min_ed = df["edition"].min()

df["next_edition"] = df["edition"].map(next_edition_map)

# lookup that checks the same team in the immediate next World Cup
lookup = (
    base[["country", "edition", "placement", "position"]]
    .rename(
        columns={
            "edition": "next_edition",
            "placement": "next_placement",
            "position": "next_position",
        }
    )
)

df = df.merge(lookup, on=["country", "next_edition"], how="left")

# missing in the immediate next edition = did not qualify
df.loc[df["next_edition"].notna() & df["next_position"].isna(), "next_placement"] = "DNQ"

# no next edition available in the dataset yet
df.loc[df["next_edition"].isna(), "next_placement"] = "TBD"

dnq_rank = base["position"].max() + 1
df["plot_position"] = df["next_position"]
df.loc[df["next_placement"].eq("DNQ"), "plot_position"] = dnq_rank

placement_map = {
    "Winner": "W",
    "Runner-up": "F",
    "Third Place": "3P",
    "Fourth Place": "4P",
    "Semi-final": "SF",
    "Quarter-final": "QF",
    "Round of 16": "R16",
    "Group Stage": "GS",
    "DNQ": "DNQ",
    "TBD": "TBD",
}

value_map = {
    "W": 1,
    "F": 2,
    "3P": 3,
    "4P": 4,
    "QF": 8,
    "R16": 16,
    "GS": 32,
    }

df["next_placement_short"] = df["next_placement"].map(placement_map)
df["next_placement_value"] = df["next_placement_short"].map(value_map)
fig = px.bar(
    df,
    x="edition",
    y="plot_position",
    hover_data=["country", "next_edition", "next_placement"],
    text="next_placement_short",
    title="Every World Cup Winners Placement the following Edition",
    color="country",

)
fig.update_yaxes(
    autorange="reversed",
    title= "Finish",
    tickvals=[1, 2, 3, 4, 8, 16, 32, dnq_rank],
    ticktext=["W", "F", "3P", "4P", "QF", "R16", "GS", "DNQ"],
    )
fig.update_xaxes(
    tickmode="array",
    tickvals=tick_vals,
    ticktext=[str(x) for x in tick_vals],
    tickangle=45,
    title= "Tournament Edition"
)

fig.update_layout(
    height=600,
    width=1000,
     title=dict(
            font=dict(size=20, color="Beige"),
            text= f"World Cup Winners Placement the following Edition since {min_ed}",
            x=0.5
        ),
    legend = dict(
        title="Team"
        ),
    font=dict(
        family="Courier New, monospace",
        size=10,
        color="Beige"
    ),
    margin=dict(l=40, r=40, t=60, b=40),
    paper_bgcolor="#0B0F0C", autosize=False, plot_bgcolor = '#0B0F0C',
    xaxis=dict(
        title="Tournament Edition",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='Beige',
        tickcolor= 'Beige',
    ),

    yaxis=dict(
        title="Final Placement",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='Beige',
        tickcolor= 'Beige',
    )
        
)

#add caption at the bottom of the chart
fig.add_annotation(
        text="Data Source: Kaggle | @cartierkut1",
        xref="paper", yref="paper",
        x=0, y=-0.1,
        showarrow=False,
        font=dict(size=10, color="Beige"),
)


fig.show(scale=3)
fig.write_image("assets/visualizations/b2b3.png", scale=2)

print(df["next_placement_short"].value_counts())
print(df["edition"].count())
print(df["next_placement_value"].mean())


next_placement_short
GS     5
QF     5
F      3
W      2
4P     2
R16    1
TBD    1
Name: count, dtype: int64
19
12.88888888888889


In [ ]:
elo = pd.read_csv("INT-World Cup/world_cup/by_confederation/conmebol/brazil/results.csv")
elo_df = elo.copy()
k = 10

#last K matches
L10 = elo_df.tail(k)

display(L10)

#average opp rating 
L10['opponent_elo_start'] = pd.to_numeric(L10['opponent_elo_start'])
avg_rtg = L10['opponent_elo_start'].sum() / k
avg_delta = L10['team_elo_delta'].sum() / k
max_rtg = L10['opponent_elo_start'].max()
min_rtg = L10['opponent_elo_start'].min()

max_row = L10.loc[L10["opponent_elo_start"] == max_rtg]
min_row = L10.loc[L10["opponent_elo_start"] == min_rtg]
print(avg_rtg)
print(max_rtg)
print(avg_delta)
display(max_row)
display(min_row)
print(avg_rtg)


,date,tournament,status,team,team_code,opponent,opponent_code,is_home,team_score,opponent_score,result,city,country,neutral,team_elo_start,opponent_elo_start,team_elo_end,opponent_elo_end,team_elo_delta
1047,2025-06-05,FIFA World Cup qualification,played,Brazil,BRA,Ecuador,ECU,False,0,0,draw,Guayaquil,Ecuador,False,1993.0,1911.0,1994.0,1910.0,1.0
1048,2025-06-10,FIFA World Cup qualification,played,Brazil,BRA,Paraguay,PRY,True,1,0,win,São Paulo,Brazil,False,1994.0,1832.0,2001.0,1825.0,7.0
1049,2025-09-04,FIFA World Cup qualification,played,Brazil,BRA,Chile,CHL,True,3,0,win,Rio de Janeiro,Brazil,False,2001.0,1689.0,2007.0,1683.0,6.0
1050,2025-09-09,FIFA World Cup qualification,played,Brazil,BRA,Bolivia,BOL,False,0,1,loss,El Alto,Bolivia,False,2007.0,1657.0,1975.0,1689.0,-32.0
1051,2025-10-10,Friendly,played,Brazil,BRA,South Korea,KOR,False,5,0,win,Seoul,South Korea,False,1975.0,1774.0,1989.0,1760.0,14.0
1052,2025-10-14,Kirin Cup,played,Brazil,BRA,Japan,JPN,False,2,3,loss,Tokyo,Japan,False,1989.0,1860.0,1978.0,1871.0,-11.0
1053,2025-11-15,Friendly,played,Brazil,BRA,Senegal,SEN,True,2,0,win,London,England,True,1978.0,1809.0,1986.0,1801.0,8.0
1054,2025-11-18,Friendly,played,Brazil,BRA,Tunisia,TUN,True,1,1,draw,Lille,France,True,1986.0,1642.0,1978.0,1650.0,-8.0
1055,2026-03-26,Friendly,played,Brazil,BRA,France,FRA,True,1,2,loss,Foxborough,United States,True,1978.0,2063.0,1970.0,2071.0,-8.0
1056,2026-03-31,Friendly,played,Brazil,BRA,Croatia,HRV,True,3,1,win,Orlando,United States,True,1970.0,1944.0,1984.0,1930.0,14.0


1818.1
2063.0
-0.9


C:\Users\oukan\AppData\Local\Temp\ipykernel_6948\3532076776.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,date,tournament,status,team,team_code,opponent,opponent_code,is_home,team_score,opponent_score,result,city,country,neutral,team_elo_start,opponent_elo_start,team_elo_end,opponent_elo_end,team_elo_delta
1055,2026-03-26,Friendly,played,Brazil,BRA,France,FRA,True,1,2,loss,Foxborough,United States,True,1978.0,2063.0,1970.0,2071.0,-8.0


,date,tournament,status,team,team_code,opponent,opponent_code,is_home,team_score,opponent_score,result,city,country,neutral,team_elo_start,opponent_elo_start,team_elo_end,opponent_elo_end,team_elo_delta
1054,2025-11-18,Friendly,played,Brazil,BRA,Tunisia,TUN,True,1,1,draw,Lille,France,True,1986.0,1642.0,1978.0,1650.0,-8.0


1818.1


In [ ]:
def add_recency_weights(L10, scheme="linear"):
    """
    Add recency weights to matches ordered from oldest to newest.

    Parameters
    ----------
    L10 : pd.DataFrame
        Match history sorted from oldest to newest.
    scheme : str
        "equal"  -> all matches have weight 1
        "linear" -> weights increase 1, 2, 3, ..., K

    Returns
    -------
    pd.DataFrame
        Copy of L10 with a new 'recency_weight' column.
    """
    elo = pd.read_csv("INT-World Cup/world_cup/by_confederation/conmebol/argentina/results.csv")
    elo_df = elo.copy()
    k = 10

    #last K matches
    L10 = elo_df.tail(k)

    if scheme == "equal":
        L10["recency_weight"] = 1
    elif scheme == "linear":
        L10["recency_weight"] = range(1, k + 1)
    else:
        raise ValueError("scheme must be 'equal' or 'linear'")

    return L10


    


def compute_form_features(L10):
    """
    Compute recent form features from the last K matches.

    Assumes the dataframe has:
    - team_elo_before
    - opp_elo_before
    - elo_delta
    - match_importance_k

    Returns a dictionary of features.
    """
    df = L10.copy()

    # Make sure matches are ordered oldest -> newest before assigning recency weights
    df = df.sort_values("date").reset_index(drop=True)

    # Add recency weights
    df = add_recency_weights(df, scheme="linear")

    # Total recency weight
    total_weight = df["recency_weight"].sum()

    # Weighted average Elo delta
    form_delta_k = (df["team_elo_delta"] * df["recency_weight"]).sum() / total_weight

    # Weighted average opponent Elo
    avg_opp_elo_k = (df["opponent_elo_start"] * df["recency_weight"]).sum() / total_weight

    # Elo gap = opponent Elo - team Elo
    df["elo_gap"] = df["opponent_elo_start"] - df["team_elo_start"]
    avg_elo_gap_k = (df["elo_gap"] * df["recency_weight"]).sum() / total_weight


    #schedule difficulty
    sched_diff = (df["elo_gap"] * df["recency_weight"]).sum() / total_weight

    # Average match importance constant over recent games
    #avg_match_importance_k = (df["match_importance_k"] * df["recency_weight"]).sum() / total_weight

    # Competition mix shares
    #friendly_share_k = (df["tournament_type"].eq("friendly")).mean()
    #qualifier_share_k = (df["tournament_type"].eq("qualifier")).mean()
    #finals_share_k = (df["tournament_type"].eq("finals")).mean()

    display(df)
    return {
        "form_delta_k": form_delta_k,
        "avg_opp_elo_k": avg_opp_elo_k,
        "avg_elo_gap_k": avg_elo_gap_k,
        "fixture_diff": sched_diff,
        #"friendly_share_k": friendly_share_k,
        #"qualifier_share_k": qualifier_share_k,
        #"finals_share_k": finals_share_k
    }

features = compute_form_features(L10)
features

C:\Users\oukan\AppData\Local\Temp\ipykernel_6948\2924120870.py:28: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,date,tournament,status,team,team_code,opponent,opponent_code,is_home,team_score,opponent_score,result,city,country,neutral,team_elo_start,opponent_elo_start,team_elo_end,opponent_elo_end,team_elo_delta,recency_weight,elo_gap
1054,2025-03-25,FIFA World Cup qualification,played,Argentina,ARG,Brazil,BRA,True,4,1,win,Buenos Aires,Argentina,False,2125.0,2009.0,2141.0,1993.0,16.0,1,-116.0
1055,2025-06-05,FIFA World Cup qualification,played,Argentina,ARG,Chile,CHL,False,1,0,win,Santiago,Chile,False,2141.0,1722.0,2146.0,1717.0,5.0,2,-419.0
1056,2025-06-10,FIFA World Cup qualification,played,Argentina,ARG,Colombia,COL,True,1,1,draw,Buenos Aires,Argentina,False,2146.0,1936.0,2132.0,1950.0,-14.0,3,-210.0
1057,2025-09-04,FIFA World Cup qualification,played,Argentina,ARG,Venezuela,NaN,True,3,0,win,Buenos Aires,Argentina,False,2132.0,1745.0,2136.0,1741.0,4.0,4,-387.0
1058,2025-09-09,FIFA World Cup qualification,played,Argentina,ARG,Ecuador,ECU,False,0,1,loss,Guayaquil,Ecuador,False,2136.0,1907.0,2109.0,1934.0,-27.0,5,-229.0
1059,2025-10-10,Friendly,played,Argentina,ARG,Venezuela,NaN,True,1,0,win,Miami Gardens,United States,True,2109.0,1717.0,2111.0,1715.0,2.0,6,-392.0
1060,2025-10-14,Friendly,played,Argentina,ARG,Puerto Rico,NaN,False,6,0,win,Fort Lauderdale,United States,True,2111.0,1133.0,2111.0,1133.0,0.0,7,-978.0
1061,2025-11-14,Friendly,played,Argentina,ARG,Angola,AGO,False,2,0,win,Luanda,Angola,False,2111.0,1567.0,2113.0,1565.0,2.0,8,-544.0
1062,2026-03-27,Friendly,played,Argentina,ARG,Mauritania,NaN,True,2,1,win,Buenos Aires,Argentina,False,2113.0,1311.0,2113.0,1311.0,0.0,9,-802.0
1063,2026-03-31,Friendly,played,Argentina,ARG,Zambia,NaN,True,5,0,win,Buenos Aires,Argentina,False,2113.0,1370.0,2113.0,1370.0,0.0,10,-743.0


{'form_delta_k': -1.9454545454545455,
 'avg_opp_elo_k': 1528.0727272727272,
 'avg_elo_gap_k': -590.4545454545455,
 'fixture_diff': -590.4545454545455}

In [ ]:
import numpy as np
import pandas as pd

teams = pd.read_csv("INT-World Cup/world_cup/2026/teams.csv")
history = all_editions.copy()

# prefer canonical_name; if no match, fall back to tournament_name
history_names = set(history["country"].dropna().unique())
teams["lookup_name"] = teams["canonical_name"].where(
    teams["canonical_name"].isin(history_names),
    teams["tournament_name"]
)

matched = teams.merge(
    history,
    left_on="lookup_name",
    right_on="country",
    how="left"
)

# recency weights by tournament order, not raw year
editions = sorted(history["edition"].dropna().unique())
edition_weight_map = {edition: (i + 1) ** 2 for i, edition in enumerate(editions)}
matched["recency_weight"] = matched["edition"].map(edition_weight_map)

result = (
    matched.groupby(["canonical_name", "tournament_name", "lookup_name"], dropna=False)
    .apply(
        lambda g: pd.Series({
            "previous_appearances": g["edition"].nunique() if g["edition"].notna().any() else 0,
            "appearance_count": (g["edition"].nunique() if g["edition"].notna().any() else 0) + 1,
            "avg_position": g["position"].mean(),
            "weighted_avg_position": np.average(
                g.loc[g["position"].notna(), "position"],
                weights=g.loc[g["position"].notna(), "recency_weight"]
            ) if g["position"].notna().any() else np.nan,
            "is_first_timer": not g["edition"].notna().any(),
        })
    )
    .reset_index()
)

result["avg_position"] = result["avg_position"].round(2)
result["weighted_avg_position"] = result["weighted_avg_position"].round(2)

display(result.sort_values(["weighted_avg_position", "canonical_name"]))


C:\Users\oukan\AppData\Local\Temp\ipykernel_6948\2182323201.py:28: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,canonical_name,tournament_name,lookup_name,previous_appearances,appearance_count,avg_position,weighted_avg_position,is_first_timer
44,Turkey,Türkiye,Türkiye,2,3,9.50,4.04,False
6,Brazil,Brazil,Brazil,22,23,6.36,6.18,False
28,Netherlands,Netherlands,Netherlands,11,12,8.45,7.09,False
1,Argentina,Argentina,Argentina,18,19,9.39,9.09,False
17,France,France,France,16,17,10.81,9.91,False
18,Germany,Germany,Germany,20,21,7.00,10.44,False
16,England,England,England,16,17,10.38,11.57,False
41,Sweden,Sweden,Sweden,12,13,10.33,12.45,False
40,Spain,Spain,Spain,16,17,13.81,15.50,False
4,Belgium,Belgium,Belgium,14,15,15.14,15.90,False
